# 1. Reinforcement Learning'e Giriş

Bu notebook, Sutton & Barto'nun "Reinforcement Learning: An Introduction" kitabının 1. bölümünü kapsar.

## İçindekiler
1. RL Nedir?
2. RL'nin Temel Elementleri
3. Agent-Environment Etkileşimi
4. Basit Bir Örnek

## 1.1 Reinforcement Learning Nedir?

**Reinforcement Learning (Pekiştirmeli Öğrenme)**, bir ajanın (agent) çevresiyle (environment) etkileşime girerek deneme-yanılma yoluyla öğrenmesidir.

### Diğer Öğrenme Paradigmalarından Farkı

| Paradigma | Özellik |
|-----------|--------|
| **Supervised Learning** | Etiketli veri, doğru cevap verilir |
| **Unsupervised Learning** | Etiketsiz veri, yapı keşfi |
| **Reinforcement Learning** | Ödül sinyali, deneme-yanılma |

RL'de:
- Doğru aksiyonlar **önceden verilmez**
- Agent **ödül** (reward) maksimizasyonu yapar
- **Exploration vs Exploitation** dengesi kritiktir

## 1.2 RL'nin Temel Elementleri

### 1. Policy (π)
Agent'ın davranış stratejisi. State'den action'a mapping.

$$\pi(a|s) = P(A_t = a | S_t = s)$$

### 2. Reward Signal (R)
Her adımda environment'ın verdiği sayısal geri bildirim.

### 3. Value Function (V)
Bir state'in uzun vadeli değeri.

$$V^\pi(s) = E_\pi[G_t | S_t = s]$$

### 4. Model (opsiyonel)
Environment'ın dinamiklerinin tahmini.

## 1.3 Agent-Environment Etkileşimi

```
    +-------+     action (a)      +-----------+
    | Agent | -----------------> | Environment|
    +-------+ <----------------- +-----------+
                state (s), reward (r)
```

Her zaman adımında:
1. Agent state'i gözlemler: $S_t$
2. Agent action seçer: $A_t$
3. Environment yeni state ve reward verir: $S_{t+1}, R_{t+1}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Basit bir RL loop örneği
class SimpleEnvironment:
    """Basit bir grid world environment."""
    
    def __init__(self, size=5):
        self.size = size
        self.goal = (size-1, size-1)  # Sağ alt köşe hedef
        self.reset()
    
    def reset(self):
        self.position = (0, 0)  # Sol üst köşeden başla
        return self.position
    
    def step(self, action):
        """Action: 0=yukarı, 1=sağ, 2=aşağı, 3=sol"""
        x, y = self.position
        
        if action == 0 and y > 0:
            y -= 1
        elif action == 1 and x < self.size - 1:
            x += 1
        elif action == 2 and y < self.size - 1:
            y += 1
        elif action == 3 and x > 0:
            x -= 1
        
        self.position = (x, y)
        
        # Reward: hedefe ulaşınca +1, diğer durumlarda -0.1
        if self.position == self.goal:
            return self.position, 1.0, True
        else:
            return self.position, -0.1, False

# Test edelim
env = SimpleEnvironment(size=5)
state = env.reset()
print(f"Başlangıç state: {state}")
print(f"Hedef: {env.goal}")

In [ ]:
class RandomAgent:
    """Rastgele aksiyon seçen basit bir agent."""
    
    def __init__(self, n_actions=4):
        self.n_actions = n_actions
    
    def select_action(self, state):
        return np.random.randint(self.n_actions)

# Agent-Environment etkileşimi
agent = RandomAgent()
env = SimpleEnvironment(size=5)

# Bir episode çalıştır
state = env.reset()
total_reward = 0
trajectory = [state]

for step in range(100):  # Max 100 adım
    action = agent.select_action(state)
    next_state, reward, done = env.step(action)
    total_reward += reward
    trajectory.append(next_state)
    
    if done:
        print(f"Hedefe {step+1} adımda ulaşıldı!")
        break
    
    state = next_state

print(f"Toplam reward: {total_reward:.2f}")

In [ ]:
def visualize_trajectory(trajectory, size=5):
    """Trajectory'yi grid üzerinde görselleştir."""
    fig, ax = plt.subplots(figsize=(6, 6))
    
    # Grid çiz
    for i in range(size + 1):
        ax.axhline(y=i, color='gray', linewidth=0.5)
        ax.axvline(x=i, color='gray', linewidth=0.5)
    
    # Trajectory çiz
    xs = [p[0] + 0.5 for p in trajectory]
    ys = [size - p[1] - 0.5 for p in trajectory]
    
    ax.plot(xs, ys, 'b-', linewidth=2, alpha=0.7)
    ax.scatter(xs[0], ys[0], color='green', s=200, zorder=5, label='Start')
    ax.scatter(xs[-1], ys[-1], color='red', s=200, zorder=5, label='End')
    
    # Hedef
    ax.add_patch(plt.Rectangle((size-1, 0), 1, 1, color='gold', alpha=0.5))
    ax.text(size-0.5, 0.5, 'GOAL', ha='center', va='center', fontsize=10)
    
    ax.set_xlim(0, size)
    ax.set_ylim(0, size)
    ax.set_aspect('equal')
    ax.legend()
    ax.set_title(f'Agent Trajectory ({len(trajectory)-1} steps)')
    plt.show()

visualize_trajectory(trajectory)

## 1.4 Return ve Discounting

Agent'ın amacı **kümülatif ödülü (return)** maksimize etmektir:

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + ... = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

Burada $\gamma \in [0, 1]$ **discount factor**'dür:
- $\gamma = 0$: Sadece anlık ödül önemli (myopic)
- $\gamma = 1$: Tüm gelecek ödüller eşit önemli
- $\gamma \approx 0.99$: Uzun vadeli düşünme

In [ ]:
def calculate_return(rewards, gamma=0.99):
    """Verilen reward listesi için discounted return hesapla."""
    G = 0
    returns = []
    
    # Sondan başa doğru hesapla (daha verimli)
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    
    return returns

# Örnek: Bir episode'daki reward'lar
rewards = [-0.1, -0.1, -0.1, -0.1, 1.0]  # 4 adım + hedefe ulaşma

for gamma in [0.0, 0.5, 0.9, 0.99, 1.0]:
    returns = calculate_return(rewards, gamma)
    print(f"γ={gamma:.2f}: Returns = {[f'{r:.3f}' for r in returns]}")

## Özet

Bu notebook'ta öğrendiklerimiz:

1. **RL**, agent'ın environment ile etkileşerek öğrendiği bir paradigmadır
2. Temel elementler: **Policy, Reward, Value Function, Model**
3. Agent her adımda state gözlemler, action seçer, reward alır
4. **Return**, gelecek ödüllerin discounted toplamıdır

### Sonraki Notebook
**02 - Multi-Armed Bandits**: Exploration vs Exploitation problemi